# Projet 15 — CNN de détection de chutes (UR Fall)

Cette version contient uniquement le workflow utile pour entraîner le CNN et exporter ses résultats pour les modèles RNN et Random Forest.

**Aucun téléchargement automatique du dataset, aucune installation automatique de vidéos/RGB.** Vous fournissez vous-même les fichiers du dataset dans Google Drive.

Exécutez les cellules dans l'ordre. Dans Colab, sélectionnez **Runtime → Change runtime type → T4 GPU** si disponible.

## 1. Vérifier l'environnement

In [ ]:
import torch
import cv2
import numpy as np
import pandas as pd
import sklearn
import matplotlib

print("PyTorch :", torch.__version__)
print("OpenCV :", cv2.__version__)
print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("Le CNN peut fonctionner sur CPU, mais l'entraînement sera plus lent.")

## 2. Charger le programme CNN

In [ ]:
%%writefile fall_cnn.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
fall_cnn.py - CNN posture classifier for Projet 15 (fall detection, UR Fall dataset).

What this script does
---------------------
1. Finds the UR Fall RGB frames (camera 0) and the official label files
   (urfall-cam0-falls.csv / urfall-cam0-adls.csv).
2. Turns every frame into a small *silhouette* image (privacy friendly: no face,
   no texture): background subtraction -> person mask -> crop around the
   person -> pad to a square (aspect ratio is kept) -> 64x64.
   Input modes (--mode):
     auto        depth if the cam0 depth frames are available, else gray   (default)
     depth       silhouette from the Kinect depth maps (most reliable: the background is
                 the farthest value ever seen at each pixel, so a person lying still
                 is NOT absorbed into the background)
     gray        whole grayscale frame resized to 64x64 (no segmentation, always works)
     silhouette  silhouette from RGB with a median background. WARNING: ghosting when the
                 person stays still for ~half of the sequence (lying phase of a fall).
3. Trains a small CNN that classifies each frame into 3 postures:
       0 not_lying  (label -1 in the dataset)
       1 falling    (label  0 in the dataset: transition / temporary pose)
       2 lying      (label  1 in the dataset)
4. Evaluates on sequences the network has never seen (split by SEQUENCE, never
   by frame, otherwise neighbouring frames leak between train and test).
5. Exports per-frame probabilities + 64-d embeddings, so that the RNN
   (routine deviations) and the Random Forest (alert decision) teammates can
   use the CNN output as input features.

Sub-commands
------------
  inspect   check the dataset layout and labels            (no PyTorch needed)
  preview   save a picture of the preprocessed frames       (no PyTorch needed)
  prepare   build the silhouette cache                      (no PyTorch needed)
  train     train + evaluate + export                       (PyTorch)
  demo      run the trained CNN on a sequence folder/video  (PyTorch)

Typical use
-----------
  python fall_cnn.py inspect --data /path/to/urfall
  python fall_cnn.py preview --data /path/to/urfall
  python fall_cnn.py train   --data /path/to/urfall
  python fall_cnn.py demo    --input /path/to/urfall/fall-01-cam0-rgb --model fall_cnn_out/model.pt
"""
from __future__ import annotations

import argparse
import copy
import json
import os
import re
import sys
import time
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------- #
# Constants
# --------------------------------------------------------------------------- #
CLASS_NAMES = ["not_lying", "falling", "lying"]
LABEL_TO_CLASS = {-1: 0, 0: 1, 1: 2}          # dataset label -> class id
CLASS_TO_LABEL = {v: k for k, v in LABEL_TO_CLASS.items()}
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp"}
SEQ_RE = re.compile(r"(fall|adl)[-_ ]?(\d+)", re.IGNORECASE)
PROC_WIDTH = 320      # frames are downscaled to this width before processing
EMB_DIM = 64
# UR Fall depth PNG16: depth_mm = C * pixel / 65535, C = 6000 (falls, cam0) / 7000 (ADLs, cam0)
DEPTH_C = {"fall": 6000.0, "adl": 7000.0}


# --------------------------------------------------------------------------- #
# Dataset discovery
# --------------------------------------------------------------------------- #
def seq_name(text: str):
    """'fall-01-cam0-rgb' -> 'fall-01' ; 'ADL_7' -> 'adl-07' ; else None."""
    m = SEQ_RE.search(text)
    return f"{m.group(1).lower()}-{int(m.group(2)):02d}" if m else None


def is_cam1(part: str) -> bool:
    return "cam1" in part.lower()


def is_depth_name(part: str) -> bool:
    p = part.lower()
    return "depth" in p or re.search(r"(^|[-_.])d$", p) is not None


def last_int(stem: str):
    nums = re.findall(r"\d+", stem)
    return int(nums[-1]) if nums else None


def extract_zips(data_dir: Path, out_dir: Path, include_depth: bool = True) -> int:
    """Extract camera-0 zips (fall-XX-cam0-rgb.zip, fall-XX-cam0-d.zip) into our output folder."""
    n = 0
    for z in sorted(Path(data_dir).rglob("*.zip")):
        if is_cam1(z.stem) or seq_name(z.stem) is None:
            continue
        if is_depth_name(z.stem) and not include_depth:
            continue
        target = Path(out_dir) / z.stem
        if target.exists() and any(target.iterdir()):
            continue
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(target)
        n += 1
    return n


def find_sequences(roots, kind: str = "rgb") -> dict:
    """Return {'fall-01': Path(dir with the cam0 frames), ...}; kind = 'rgb' or 'depth'."""
    found = {}
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for dirpath, _dirs, files in os.walk(root):
            imgs = [f for f in files if Path(f).suffix.lower() in IMG_EXT]
            if not imgs:
                continue
            rel = Path(dirpath).relative_to(root).parts
            if not rel:
                continue
            if any(is_cam1(p) for p in rel):
                continue
            has_depth = any(is_depth_name(p) for p in rel)
            if has_depth != (kind == "depth"):
                continue
            seq = next((s for s in (seq_name(p) for p in reversed(rel)) if s), None)
            if seq is None:
                continue
            if seq not in found or len(imgs) > found[seq][1]:
                found[seq] = (Path(dirpath), len(imgs))
    return {k: v[0] for k, v in found.items()}


def list_frames(d: Path):
    """[(frame_number, path), ...] sorted by frame number (duplicates dropped)."""
    items = {}
    for f in sorted(Path(d).iterdir()):
        if f.suffix.lower() in IMG_EXT:
            n = last_int(f.stem)
            if n is not None and n not in items:
                items[n] = f
    return sorted(items.items())


def find_label_csvs(data_dir: Path):
    out = []
    for p in sorted(Path(data_dir).rglob("*.csv")):
        n = p.name.lower()
        if "urfall" in n and "cam1" not in n:
            out.append(p)
    return out


def _is_number(x) -> bool:
    try:
        float(x)
        return True
    except (TypeError, ValueError):
        return False


def load_labels(paths) -> pd.DataFrame:
    """Read the official UR Fall label CSVs -> DataFrame[seq, frame, label].
    Columns used (by position): 0 = sequence, 1 = frame number, 2 = label.
    Works with or without a header row."""
    parts = []
    for p in paths:
        df = pd.read_csv(p, header=None, dtype=str)
        if df.shape[1] < 3:
            df = pd.read_csv(p, header=None, dtype=str, sep=";")
        if df.shape[1] < 3:
            raise ValueError(f"{p}: expected >= 3 columns (sequence, frame, label)")
        if len(df) and not _is_number(df.iloc[0, 1]):
            df = df.iloc[1:]                       # header row
        d = pd.DataFrame({
            "seq": [seq_name(str(s)) for s in df.iloc[:, 0]],
            "frame": pd.to_numeric(df.iloc[:, 1], errors="coerce"),
            "label": pd.to_numeric(df.iloc[:, 2], errors="coerce"),
        })
        d = d.dropna()
        d = d[d["label"].isin([-1, 0, 1])]
        d["frame"] = d["frame"].astype(int)
        d["label"] = d["label"].astype(int)
        parts.append(d)
    if not parts:
        return pd.DataFrame(columns=["seq", "frame", "label"])
    return pd.concat(parts, ignore_index=True)


class Index:
    """Result of build_index(): which image belongs to which label."""
    def __init__(self):
        self.records = pd.DataFrame()
        self.info = {}


def discover(args):
    """Extract zips if needed and locate the cam0 RGB and depth frame folders."""
    data, out = Path(args.data), Path(args.out)
    n_zip = 0
    if data.exists():
        n_zip = extract_zips(data, out / "extracted", include_depth=args.mode in ("auto", "depth"))
    roots = [data, out / "extracted"]
    return find_sequences(roots, "rgb"), find_sequences(roots, "depth"), n_zip


def resolve_mode(args, rgb: dict, dep: dict) -> str:
    """auto -> depth when (almost) every sequence has depth frames, else gray. Mutates args.mode."""
    if args.mode == "auto":
        args.mode = "depth" if dep and len(dep) >= 0.8 * len(rgb) else "gray"
    return args.mode


def ensure_mode(args) -> str:
    rgb, dep, _ = discover(args)
    return resolve_mode(args, rgb, dep)


def build_index(args) -> Index:
    data, out = Path(args.data), Path(args.out)
    idx = Index()
    info = idx.info
    info["requested_mode"] = args.mode
    rgb_dirs, depth_dirs, info["zips_extracted"] = discover(args)
    resolve_mode(args, rgb_dirs, depth_dirs)
    info["mode"] = args.mode
    info["n_rgb_seqs"], info["n_depth_seqs"] = len(rgb_dirs), len(depth_dirs)
    seq_dirs = depth_dirs if args.mode == "depth" else rgb_dirs
    csvs = [Path(p) for p in args.labels] if args.labels else find_label_csvs(data)
    info["csv_files"] = [str(c) for c in csvs]
    info["seq_dirs"] = {k: str(v) for k, v in seq_dirs.items()}
    lab = load_labels(csvs) if csvs else pd.DataFrame(columns=["seq", "frame", "label"])
    labels_by_seq = {s: dict(zip(g["frame"].tolist(), g["label"].tolist()))
                     for s, g in lab.groupby("seq")} if len(lab) else {}
    frames_by_seq = {s: list_frames(d) for s, d in seq_dirs.items()}

    # Some copies number frames from 0, others from 1: pick the offset that matches best.
    best_off, best_hits = 0, -1
    for off in (0, 1, -1, 2, -2):
        hits = sum(1 for s, fr in frames_by_seq.items() for n, _ in fr
                   if (n + off) in labels_by_seq.get(s, {}))
        if hits > best_hits:
            best_off, best_hits = off, hits
    info["frame_offset"] = best_off

    common = sorted(set(frames_by_seq) & set(labels_by_seq))
    if getattr(args, "max_seqs", 0):
        keep = []
        for kind in ("fall", "adl"):
            keep += [s for s in common if s.startswith(kind)][: args.max_seqs]
        common = keep
    rows = []
    for s in common:
        lab_s = labels_by_seq[s]
        for n, path in frames_by_seq[s]:
            f = n + best_off
            if f in lab_s:
                rows.append((s, s.split("-")[0], str(path), n, f, lab_s[f]))
    idx.records = pd.DataFrame(rows, columns=["seq", "kind", "path", "img_frame", "frame", "label"])
    info["n_images"] = int(sum(len(v) for v in frames_by_seq.values()))
    info["n_labeled_images"] = len(idx.records)
    info["seqs_without_labels"] = sorted(set(frames_by_seq) - set(labels_by_seq))
    info["seqs_without_images"] = sorted(set(labels_by_seq) - set(frames_by_seq))
    info["n_label_rows"] = int(len(lab))
    return idx


# --------------------------------------------------------------------------- #
# Preprocessing: frame -> silhouette
# --------------------------------------------------------------------------- #
def to_proc_gray(img) -> np.ndarray:
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    h, w = g.shape[:2]
    if w > PROC_WIDTH:
        g = cv2.resize(g, (PROC_WIDTH, int(round(h * PROC_WIDTH / w))), interpolation=cv2.INTER_AREA)
    return g


def read_proc_gray(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    return None if img is None else to_proc_gray(img)


def depth_to_mm(raw, seq_kind: str, scale_c=None) -> np.ndarray:
    """Raw depth PNG array -> float32 millimetres at PROC_WIDTH (0 = invalid)."""
    if raw.ndim == 3:
        raw = raw[..., 0]
    maxval = 255.0 if raw.dtype == np.uint8 else 65535.0
    c = scale_c if scale_c else DEPTH_C.get(seq_kind, 6000.0)
    mm = raw.astype(np.float32) * (c / maxval)
    h, w = mm.shape[:2]
    if w > PROC_WIDTH:     # nearest neighbour: never blend valid depth with invalid zeros
        mm = cv2.resize(mm, (PROC_WIDTH, int(round(h * PROC_WIDTH / w))), interpolation=cv2.INTER_NEAREST)
    return mm


def read_proc_depth(path, seq_kind: str, scale_c=None):
    raw = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    return None if raw is None else depth_to_mm(raw, seq_kind, scale_c)


def read_proc(path, mode: str, seq_kind: str):
    """One frame -> processing array: float32 mm for depth mode, uint8 gray otherwise."""
    return read_proc_depth(path, seq_kind) if mode == "depth" else read_proc_gray(path)


def depth_vis(mm) -> np.ndarray:
    """Depth (mm) -> uint8 picture (near = bright) for previews / demo overlays."""
    v = np.clip(mm / 6000.0, 0, 1)
    out = (255 * (1 - v)).astype(np.uint8)
    out[mm <= 0] = 0
    return out


def median_background(grays, n_samples=40) -> np.ndarray:
    ids = np.linspace(0, len(grays) - 1, min(len(grays), n_samples)).astype(int)
    return np.median(np.stack([grays[i] for i in ids]), axis=0).astype(np.uint8)


def silhouette(gray, bg, thr=30, min_area=150):
    """Binary person mask (uint8 0/255) or None if nobody is found."""
    g = cv2.GaussianBlur(gray, (5, 5), 0)
    b = cv2.GaussianBlur(bg, (5, 5), 0)
    diff = cv2.absdiff(g, b)
    _, m = cv2.threshold(diff, thr, 255, cv2.THRESH_BINARY)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
    return largest_blobs(m, min_area)


def largest_blobs(m, min_area):
    """Keep the largest connected blob plus blobs >= 25% of its area (a person split in pieces)."""
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n <= 1:
        return None
    areas = stats[1:, cv2.CC_STAT_AREA]
    amax = int(areas.max())
    if amax < min_area:
        return None
    keep = [i + 1 for i, a in enumerate(areas) if a >= max(min_area, 0.25 * amax)]
    return np.isin(lab, keep).astype(np.uint8) * 255


def to_square(mask, size, margin=0.08) -> np.ndarray:
    """Crop around the person, pad to a square (keeps the aspect ratio), resize."""
    ys, xs = np.where(mask > 0)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    crop = mask[y0:y1, x0:x1]
    h, w = crop.shape
    side = max(int(round(max(h, w) * (1 + 2 * margin))), 2)
    canvas = np.zeros((side, side), np.uint8)
    oy, ox = (side - h) // 2, (side - w) // 2
    canvas[oy:oy + h, ox:ox + w] = crop
    return cv2.resize(canvas, (size, size), interpolation=cv2.INTER_AREA)


def depth_background(depths, n_samples=30, percentile=80) -> np.ndarray:
    """Per-pixel background = a high percentile of the valid depths seen over the sequence.
    The person is always CLOSER than the background, so even a person who lies still for most
    of the sequence cannot become part of it as long as each pixel is free in a few frames."""
    ids = np.linspace(0, len(depths) - 1, min(len(depths), n_samples)).astype(int)
    st = np.stack([depths[i] for i in ids]).astype(np.float32)
    valid = st > 0
    st = np.where(valid, st, np.inf)          # invalid pixels sort to the end
    st.sort(axis=0)
    cnt = valid.sum(axis=0)                   # number of valid samples per pixel
    k = np.clip(np.floor(percentile / 100.0 * (cnt - 1)).astype(np.int64), 0, None)
    bg = np.take_along_axis(st, k[None], axis=0)[0]
    bg[cnt == 0] = 0.0                        # never valid -> no background -> never foreground
    return bg.astype(np.float32)


def depth_silhouette(mm, bg, thr_mm=120.0, min_area=150):
    """Binary person mask (uint8 0/255) from a depth map, or None."""
    valid = (mm > 0) & (bg > 0)
    m = (valid & ((bg - mm) > thr_mm)).astype(np.uint8) * 255
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
    return largest_blobs(m, min_area)


def preprocess_frames(grays, mode, size, thr=30, min_area=150, depth_thr=120.0):
    """One sequence -> (X uint8 [n,size,size], found bool [n]).
    `grays` = uint8 gray frames (modes gray/silhouette) or float32 mm depth maps (mode depth)."""
    n = len(grays)
    X = np.zeros((n, size, size), np.uint8)
    found = np.zeros(n, bool)
    if n == 0:
        return X, found
    shape = grays[0].shape
    interp = cv2.INTER_NEAREST if mode == "depth" else cv2.INTER_LINEAR
    grays = [g if g.shape == shape else cv2.resize(g, (shape[1], shape[0]), interpolation=interp)
             for g in grays]
    if mode == "depth":
        bg = depth_background(grays)
        for i, d in enumerate(grays):
            m = depth_silhouette(d, bg, depth_thr, min_area)
            if m is not None:
                X[i] = to_square(m, size)
                found[i] = True
        return X, found
    if mode == "gray":
        for i, g in enumerate(grays):
            X[i] = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA)
            found[i] = True
        return X, found
    bg = median_background(grays)
    for i, g in enumerate(grays):
        m = silhouette(g, bg, thr, min_area)
        if m is not None:
            X[i] = to_square(m, size)
            found[i] = True
    return X, found


# --------------------------------------------------------------------------- #
# Cache
# --------------------------------------------------------------------------- #
def cache_path(args) -> Path:
    tag = f"frames_{args.mode}_{args.size}_t{args.thr}_d{args.depth_thr:g}_a{args.min_area}"
    if getattr(args, "max_seqs", 0):
        tag += f"_max{args.max_seqs}"
    return Path(args.out) / "cache" / f"{tag}.npz"


def prepare_cache(args) -> dict:
    ensure_mode(args)                       # resolves --mode auto
    print(f"[prepare] input mode: {args.mode}")
    cp = cache_path(args)
    if cp.exists() and not args.force:
        print(f"[prepare] using cache {cp}")
        z = np.load(cp, allow_pickle=False)
        return {k: z[k] for k in z.files}
    idx = build_index(args)
    rec = idx.records
    if rec.empty:
        sys.exit("No labeled frames found. Run:  python fall_cnn.py inspect --data <folder>")
    Xs, ys, seqs, frames, labels, founds = [], [], [], [], [], []
    t0 = time.time()
    groups = list(rec.groupby("seq", sort=True))
    for k, (seq, g) in enumerate(groups, 1):
        g = g.sort_values("frame")
        grays, keep = [], []
        for pos, p in enumerate(g["path"].tolist()):
            gr = read_proc(p, args.mode, seq.split("-")[0])
            if gr is not None:
                grays.append(gr)
                keep.append(pos)
        g = g.iloc[keep]
        X, f = preprocess_frames(grays, args.mode, args.size, args.thr, args.min_area, args.depth_thr)
        Xs.append(X)
        founds.append(f)
        ys += [LABEL_TO_CLASS[int(l)] for l in g["label"]]
        labels += g["label"].astype(int).tolist()
        seqs += [seq] * len(g)
        frames += g["frame"].astype(int).tolist()
        print(f"[prepare] {k:3d}/{len(groups)} {seq}: {len(g)} frames, "
              f"silhouette found in {int(f.sum())}  ({time.time() - t0:.0f}s)", flush=True)
    data = {
        "X": np.concatenate(Xs),
        "y": np.array(ys, np.int64),
        "seq": np.array(seqs),
        "frame": np.array(frames, np.int64),
        "label": np.array(labels, np.int64),
        "found": np.concatenate(founds),
    }
    cp.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cp, **data)
    print(f"[prepare] saved {cp}  ({len(data['y'])} frames)")
    return data


# --------------------------------------------------------------------------- #
# Split by sequence
# --------------------------------------------------------------------------- #
def split_sequences(seqs, seed=42, val_frac=0.15, test_frac=0.15) -> dict:
    """Stratified (falls / ADLs separately) split of whole sequences."""
    rng = np.random.RandomState(seed)
    out = {}
    for kind in ("fall", "adl"):
        s = sorted(x for x in seqs if x.startswith(kind))
        rng.shuffle(s)
        n = len(s)
        nt, nv = int(round(n * test_frac)), int(round(n * val_frac))
        if n >= 3:
            nt, nv = max(nt, 1), max(nv, 1)
        for i, name in enumerate(s):
            out[name] = "test" if i < nt else "val" if i < nt + nv else "train"
    return out


# --------------------------------------------------------------------------- #
# Model (PyTorch is imported lazily so inspect/preview/prepare work without it)
# --------------------------------------------------------------------------- #
def make_model(n_classes: int, emb_dim: int = EMB_DIM):
    import torch.nn as nn

    class FallCNN(nn.Module):
        """4 conv blocks (16-32-64-128) -> global pooling -> 64-d embedding -> classes."""

        def __init__(self):
            super().__init__()

            def block(i, o):
                return nn.Sequential(
                    nn.Conv2d(i, o, 3, padding=1, bias=False),
                    nn.BatchNorm2d(o),
                    nn.ReLU(inplace=True),
                    nn.MaxPool2d(2),
                )

            self.features = nn.Sequential(
                block(1, 16), block(16, 32), block(32, 64), block(64, 128),
                nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            )
            self.embed = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, emb_dim), nn.ReLU(inplace=True))
            self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(emb_dim, n_classes))

        def forward(self, x, return_emb=False):
            e = self.embed(self.features(x))
            logits = self.head(e)
            return (logits, e) if return_emb else logits

    return FallCNN()


def augment(x):
    """Random horizontal flip + small zoom + small shift (on a batch, on the GPU/CPU)."""
    import torch
    import torch.nn.functional as F
    B, dev = x.size(0), x.device
    flip = 1.0 - 2.0 * (torch.rand(B, device=dev) < 0.5).float()      # +1 keep, -1 flip
    s = 1.0 + (torch.rand(B, device=dev) - 0.5) * 0.2                 # 0.9 .. 1.1
    tx = (torch.rand(B, device=dev) - 0.5) * 0.16
    ty = (torch.rand(B, device=dev) - 0.5) * 0.16
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0] = s * flip
    theta[:, 0, 2] = tx
    theta[:, 1, 1] = s
    theta[:, 1, 2] = ty
    grid = F.affine_grid(theta, list(x.size()), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)


def load_model(path, device):
    import torch
    ckpt = torch.load(path, map_location=device, weights_only=True)
    model = make_model(ckpt["n_classes"], ckpt["emb_dim"])
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()
    return model, ckpt


def run_model(model, X, device, bs=256):
    """X uint8 [n,S,S] -> probs [n,K], embeddings [n,EMB]."""
    import torch
    Xt = torch.from_numpy(np.ascontiguousarray(X)).unsqueeze(1)
    P, E = [], []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(Xt), bs):
            xb = Xt[i:i + bs].to(device).float().div_(255.0)
            lo, em = model(xb, return_emb=True)
            P.append(torch.softmax(lo, 1).cpu().numpy())
            E.append(em.cpu().numpy())
    if not P:
        return np.zeros((0, len(CLASS_NAMES)), np.float32), np.zeros((0, EMB_DIM), np.float32)
    return np.concatenate(P), np.concatenate(E)


# --------------------------------------------------------------------------- #
# Reports (no torch needed)
# --------------------------------------------------------------------------- #
def write_reports(out: Path, history, y_true, y_pred, cfg, split_map):
    from sklearn.metrics import classification_report, confusion_matrix
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    K = len(CLASS_NAMES)
    labels = list(range(K))
    text = classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES, zero_division=0)
    rep = classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES,
                                zero_division=0, output_dict=True)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    (out / "test_report.txt").write_text(text + "\nConfusion matrix (rows=true, cols=pred):\n" + str(cm) + "\n")
    (out / "metrics.json").write_text(json.dumps(
        {"config": cfg, "test_report": rep, "confusion_matrix": cm.tolist(),
         "history": history, "split": split_map}, indent=2))
    print("\n=== TEST SET (sequences never seen during training) ===")
    print(text)
    print("Confusion matrix (rows=true, cols=pred):\n", cm)

    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(labels)
    ax.set_xticklabels(CLASS_NAMES, rotation=30)
    ax.set_yticks(labels)
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    top = cm.max() if cm.size else 1
    for i in range(K):
        for j in range(K):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > top / 2 else "black")
    ax.set_title("Confusion matrix (test)")
    fig.tight_layout()
    fig.savefig(out / "confusion_matrix.png", dpi=150)
    plt.close(fig)

    if history:
        ep = [h["epoch"] for h in history]
        fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
        axs[0].plot(ep, [h["train_loss"] for h in history], label="train")
        axs[0].plot(ep, [h["val_loss"] for h in history], label="val")
        axs[0].set_xlabel("epoch")
        axs[0].set_ylabel("loss")
        axs[0].legend()
        axs[1].plot(ep, [h["val_macro_f1"] for h in history], label="val macro-F1")
        axs[1].plot(ep, [h["val_acc"] for h in history], label="val accuracy")
        axs[1].set_xlabel("epoch")
        axs[1].legend()
        fig.tight_layout()
        fig.savefig(out / "training_curves.png", dpi=150)
        plt.close(fig)


# --------------------------------------------------------------------------- #
# Commands
# --------------------------------------------------------------------------- #
def cmd_inspect(args) -> int:
    data = Path(args.data)
    if not data.exists():
        print(f"ERROR: folder not found: {data}")
        return 1
    idx = build_index(args)
    info, rec = idx.info, idx.records
    sd = info["seq_dirs"]
    nf = sum(1 for s in sd if s.startswith("fall"))
    na = sum(1 for s in sd if s.startswith("adl"))
    print(f"Data folder              : {data}")
    print(f"Zips extracted this run  : {info['zips_extracted']}")
    print(f"Sequences with {'depth' if info['mode'] == 'depth' else 'RGB'} frames: {len(sd)}  "
          f"(falls: {nf}, ADLs: {na})   [UR Fall has 30 + 40]")
    print(f"RGB sequences found      : {info['n_rgb_seqs']}     depth sequences found: {info['n_depth_seqs']}")
    print(f"Input mode that will be used (--mode {info['requested_mode']}): {info['mode']}")
    if info["mode"] == "gray" and info["requested_mode"] == "auto":
        print("   (no depth frames found -> using plain grayscale frames. If you have the "
              "fall-XX-cam0-d.zip / adl-XX-cam0-d.zip files, put them in the data folder for much cleaner silhouettes.)")
    print(f"Label CSV files          : {len(info['csv_files'])}")
    for c in info["csv_files"]:
        print(f"   - {c}")
    print(f"Label rows               : {info['n_label_rows']}")
    print(f"Images found             : {info['n_images']}")
    print(f"Images with a label      : {info['n_labeled_images']}   (frame offset used: {info['frame_offset']})")
    if len(rec):
        print("\nClass distribution (dataset label -> class):")
        for kind in ("fall", "adl"):
            sub = rec[rec.kind == kind]
            cnt = {l: int((sub.label == l).sum()) for l in (-1, 0, 1)}
            print(f"   {kind:4s}: not lying(-1)={cnt[-1]:6d}   falling(0)={cnt[0]:5d}   lying(1)={cnt[1]:6d}")
    problems = []
    if not sd:
        problems.append("No camera-0 frame folders/zips were found. Expected e.g. fall-01-cam0-rgb/*.png and "
                        "fall-01-cam0-d/*.png (or the .zip files).")
        try:
            names = sorted(p.name for p in data.iterdir())[:15]
            print(f"\nTop of {data}: {names}")
        except OSError:
            pass
    if not info["csv_files"]:
        problems.append("No label CSV found (urfall-cam0-falls.csv / urfall-cam0-adls.csv). Download them from the "
                        "UR Fall page, put them in the data folder, or pass --labels file1.csv file2.csv")
    if sd and info["csv_files"] and info["n_images"]:
        rate = info["n_labeled_images"] / info["n_images"]
        if rate < 0.8:
            problems.append(f"Only {rate:.0%} of the images have a label: frame numbering probably does not match.")
    if info["seqs_without_labels"]:
        print(f"\nWarning: sequences with frames but no labels: {info['seqs_without_labels']}")
    if info["seqs_without_images"]:
        print(f"Warning: sequences with labels but no frames   : {info['seqs_without_images']}")
    if problems:
        print("\nPROBLEMS:")
        for p in problems:
            print(" -", p)
        return 1
    print("\nOK - the dataset looks usable. Next:  python fall_cnn.py preview ...")
    return 0


def cmd_preview(args) -> int:
    idx = build_index(args)
    rec = idx.records
    if rec.empty:
        print("No labeled frames. Run `inspect` first.")
        return 1
    rng = np.random.RandomState(args.seed)
    if args.seq:
        seqs = [seq_name(s) for s in args.seq]
    else:
        falls = sorted(s for s in rec.seq.unique() if s.startswith("fall"))
        adls = sorted(s for s in rec.seq.unique() if s.startswith("adl"))
        seqs = list(rng.choice(falls, min(2, len(falls)), replace=False)) if falls else []
        seqs += list(rng.choice(adls, min(1, len(adls)), replace=False)) if adls else []
    rows = []
    for seq in seqs:
        g = rec[rec.seq == seq].sort_values("frame")
        if g.empty:
            print(f"(skipping {seq}: no frames)")
            continue
        grays = [read_proc(p, args.mode, seq.split("-")[0]) for p in g.path]
        ok = [i for i, x in enumerate(grays) if x is not None]
        grays = [grays[i] for i in ok]
        g = g.iloc[ok]
        X, found = preprocess_frames(grays, args.mode, args.size, args.thr, args.min_area, args.depth_thr)
        picks = np.linspace(0, len(g) - 1, 10).astype(int)
        tiles = []
        for i in picks:
            shown = depth_vis(grays[i]) if args.mode == "depth" else grays[i]
            top = cv2.resize(shown, (160, 120))
            bot = cv2.resize(X[i], (160, 160), interpolation=cv2.INTER_NEAREST)
            lab = int(g.iloc[i]["label"])
            txt = f"{int(g.iloc[i]['frame'])} {CLASS_NAMES[LABEL_TO_CLASS[lab]]}"
            cv2.putText(bot, txt, (4, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.4, 255, 1, cv2.LINE_AA)
            if not found[i]:
                cv2.putText(bot, "NO PERSON", (4, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.4, 255, 1, cv2.LINE_AA)
            tiles.append(np.vstack([top, bot]))
        row = np.hstack(tiles)
        cv2.putText(row, seq, (4, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.5, 255, 1, cv2.LINE_AA)
        rows.append(row)
        print(f"{seq}: silhouette found in {int(found.sum())}/{len(found)} frames")
    if not rows:
        return 1
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    path = out / "preview.png"
    cv2.imwrite(str(path), np.vstack(rows))
    print(f"Saved {path}  -> open it: top = original frames, bottom = what the CNN will see.")
    return 0


def cmd_prepare(args) -> int:
    prepare_cache(args)
    return 0


def cmd_train(args) -> int:
    try:
        import torch
    except ImportError:
        sys.exit("PyTorch is not installed. Install it (pip install torch) or run this on Google Colab.")
    from sklearn.metrics import f1_score

    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    print(f"[train] device: {device}")

    data = prepare_cache(args)
    X, y, seqs, found = data["X"], data["y"].astype(np.int64), data["seq"], data["found"]
    K = len(CLASS_NAMES)
    split_map = split_sequences(sorted(set(seqs.tolist())), args.seed)
    split_arr = np.array([split_map[s] for s in seqs.tolist()])
    idx = {k: np.where((split_arr == k) & found)[0] for k in ("train", "val", "test")}
    for k in idx:
        ss = sorted(s for s, v in split_map.items() if v == k)
        print(f"[train] {k:5s}: {len(ss):2d} sequences, {len(idx[k]):6d} frames "
              f"(class counts {np.bincount(y[idx[k]], minlength=K).tolist()})")
    if len(idx["train"]) == 0 or len(idx["val"]) == 0:
        sys.exit("Not enough sequences for a train/val split. Check `inspect`.")
    if int(found.sum()) < 0.8 * len(found):
        print(f"[train] WARNING: silhouette found in only {found.mean():.0%} of frames. "
              f"Run `preview` and check the preprocessing (try --depth-thr 80 / --thr 20, or --mode gray).")

    Xt = torch.from_numpy(X).unsqueeze(1)          # uint8 [N,1,S,S]
    yt = torch.from_numpy(y)

    def get(ix):
        return Xt[torch.from_numpy(ix)].to(device).float().div_(255.0)

    model = make_model(K, EMB_DIM).to(device)
    counts = np.bincount(y[idx["train"]], minlength=K).astype(np.float64)
    w = np.minimum(counts.sum() / (K * np.maximum(counts, 1)), 10.0)
    print(f"[train] class weights: {np.round(w, 2).tolist()}")
    crit = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(args.epochs, 1))

    def predict(ix):
        return run_model(model, X[ix], device)

    history, best_f1, best_state, best_ep, bad = [], -1.0, None, 0, 0
    for ep in range(1, args.epochs + 1):
        t0 = time.time()
        model.train()
        perm = np.random.permutation(idx["train"])
        tot, n = 0.0, 0
        for i in range(0, len(perm), args.batch):
            b = perm[i:i + args.batch]
            if len(b) < 2:                     # BatchNorm needs >= 2 samples
                continue
            xb = augment(get(b))
            yb = yt[torch.from_numpy(b)].to(device)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            tot += loss.item() * len(b)
            n += len(b)
        sched.step()
        P, _ = predict(idx["val"])
        yv = y[idx["val"]]
        pv = P.argmax(1)
        vloss = float(-np.log(np.clip(P[np.arange(len(yv)), yv], 1e-8, 1.0)).mean())
        vacc = float((pv == yv).mean())
        vf1 = float(f1_score(yv, pv, average="macro", labels=np.unique(yv), zero_division=0))
        history.append({"epoch": ep, "train_loss": tot / max(n, 1), "val_loss": vloss,
                        "val_acc": vacc, "val_macro_f1": vf1})
        star = ""
        if vf1 > best_f1:
            best_f1, best_ep, bad = vf1, ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            star = "  *best*"
        else:
            bad += 1
        print(f"[train] epoch {ep:3d}/{args.epochs}  loss {tot / max(n, 1):.4f}  val_loss {vloss:.4f}  "
              f"val_acc {vacc:.3f}  val_macroF1 {vf1:.3f}  ({time.time() - t0:.0f}s){star}", flush=True)
        if bad >= args.patience:
            print(f"[train] early stopping (no improvement for {args.patience} epochs)")
            break

    model.load_state_dict(best_state)
    print(f"[train] best epoch: {best_ep} (val macro-F1 {best_f1:.3f})")
    cfg = {"mode": args.mode, "size": args.size, "thr": args.thr, "depth_thr": args.depth_thr,
           "min_area": args.min_area, "seed": args.seed, "epochs_run": len(history), "best_epoch": best_ep,
           "lr": args.lr, "batch": args.batch, "classes": CLASS_NAMES}
    torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                "n_classes": K, "emb_dim": EMB_DIM, "classes": CLASS_NAMES,
                "mode": args.mode, "size": args.size, "thr": args.thr, "depth_thr": args.depth_thr,
                "min_area": args.min_area},
               out / "model.pt")

    # ---- test evaluation (unseen sequences) ----
    if len(idx["test"]):
        Pt, _ = predict(idx["test"])
        write_reports(out, history, y[idx["test"]], Pt.argmax(1), cfg, split_map)
    else:
        print("[train] no test sequences; skipping test report")

    # ---- export per-frame outputs for the RNN / Random Forest teammates ----
    P_all, E_all = predict(np.arange(len(y)))
    df = pd.DataFrame({
        "sequence": seqs, "frame": data["frame"],
        "kind": [s.split("-")[0] for s in seqs.tolist()],
        "split": split_arr, "silhouette_found": found,
        "true_label": data["label"], "true_class": [CLASS_NAMES[c] for c in y],
        "pred_class": [CLASS_NAMES[c] for c in P_all.argmax(1)],
        "p_not_lying": P_all[:, 0], "p_falling": P_all[:, 1], "p_lying": P_all[:, 2],
    })
    df.to_csv(out / "frame_predictions.csv", index=False, float_format="%.5f")
    np.savez_compressed(out / "embeddings.npz", embeddings=E_all.astype(np.float32),
                        sequence=seqs, frame=data["frame"], split=split_arr)
    print(f"\n[train] DONE. Everything is in: {out.resolve()}")
    for f in ("model.pt", "test_report.txt", "metrics.json", "confusion_matrix.png",
              "training_curves.png", "frame_predictions.csv", "embeddings.npz"):
        if (out / f).exists():
            print(f"   - {f}")
    return 0


# ---- demo -----------------------------------------------------------------
def load_input_frames(path, max_frames=3000):
    """Folder of images or a video file -> (list of BGR frames, fps or None)."""
    p = Path(path)
    frames, fps = [], None
    if p.is_dir():
        for _n, f in list_frames(p)[:max_frames]:
            im = cv2.imread(str(f), cv2.IMREAD_COLOR)
            if im is not None:
                frames.append(im)
    else:
        cap = cv2.VideoCapture(str(p))
        fps = cap.get(cv2.CAP_PROP_FPS) or None
        while len(frames) < max_frames:
            ok, im = cap.read()
            if not ok:
                break
            frames.append(im)
        cap.release()
    return frames, fps


def load_depth_frames(path, scale_c=6000.0, max_frames=3000):
    """Folder of depth PNG16 -> list of float32 mm maps (sorted by frame number)."""
    out = []
    for _n, f in list_frames(Path(path))[:max_frames]:
        d = read_proc_depth(f, "fall", scale_c)
        if d is not None:
            out.append(d)
    return out


def render_demo(frames, X, found, probs, out: Path, fps):
    """Write annotated video + CSV + probability timeline. No torch needed."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    out.mkdir(parents=True, exist_ok=True)
    pred = probs.argmax(1)
    df = pd.DataFrame({"frame_index": np.arange(len(frames)), "silhouette_found": found,
                       "pred_class": [CLASS_NAMES[c] for c in pred],
                       "p_not_lying": probs[:, 0], "p_falling": probs[:, 1], "p_lying": probs[:, 2]})
    df.to_csv(out / "demo_predictions.csv", index=False, float_format="%.5f")

    colors = {0: (0, 170, 0), 1: (0, 140, 255), 2: (0, 0, 220)}      # BGR
    h, w = frames[0].shape[:2]
    vw = cv2.VideoWriter(str(out / "demo.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps or 15.0, (w, h))
    if not vw.isOpened():
        raise RuntimeError("OpenCV could not open a video writer (mp4v). Use demo_predictions.csv instead.")
    for i, fr in enumerate(frames):
        im = cv2.resize(fr, (w, h)) if fr.shape[:2] != (h, w) else fr.copy()
        c = int(pred[i])
        col = colors[c]
        label = CLASS_NAMES[c] if found[i] else "no person"
        cv2.rectangle(im, (0, 0), (w - 1, h - 1), col, 4)
        cv2.rectangle(im, (0, 0), (min(w, 330), 34), col, -1)
        cv2.putText(im, f"{label}  {probs[i, c]:.2f}", (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                    (255, 255, 255), 2, cv2.LINE_AA)
        s = 96
        inset = cv2.cvtColor(cv2.resize(X[i], (s, s), interpolation=cv2.INTER_NEAREST), cv2.COLOR_GRAY2BGR)
        if w > s + 8 and h > s + 8:
            im[8:8 + s, w - s - 8:w - 8] = inset
        vw.write(im)
    vw.release()

    fig, ax = plt.subplots(figsize=(9, 3))
    for k, name in enumerate(CLASS_NAMES):
        ax.plot(probs[:, k], label=name)
    ax.set_xlabel("frame")
    ax.set_ylabel("probability")
    ax.set_ylim(-0.02, 1.02)
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(out / "timeline.png", dpi=150)
    plt.close(fig)
    return df


def cmd_demo(args) -> int:
    try:
        import torch
    except ImportError:
        sys.exit("PyTorch is not installed. Install it (pip install torch) or run this on Google Colab.")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, ck = load_model(args.model, device)
    mode = ck["mode"]
    frames, fps = load_input_frames(args.input, args.max_frames) if args.input else ([], None)
    if mode == "depth":
        if not args.depth_input:
            sys.exit("This model was trained on depth silhouettes: also pass --depth-input <folder with the "
                     "cam0 depth PNGs of the same sequence> (and --input <RGB folder/video> for the overlay).")
        proc = load_depth_frames(args.depth_input, args.depth_scale, args.max_frames)
        if not frames:
            frames = [cv2.cvtColor(depth_vis(d), cv2.COLOR_GRAY2BGR) for d in proc]
    else:
        if not frames:
            sys.exit("Please pass --input (folder of frames or a video file).")
        proc = [to_proc_gray(f) for f in frames]
    if not proc or not frames:
        sys.exit("No frames could be read.")
    n = min(len(frames), len(proc))
    if len(frames) != len(proc):
        print(f"WARNING: {len(frames)} display frames vs {len(proc)} model frames; using the first {n}.")
    frames, proc = frames[:n], proc[:n]
    X, found = preprocess_frames(proc, mode, ck["size"], ck["thr"], ck["min_area"], ck.get("depth_thr", 120.0))
    probs, _ = run_model(model, X, device)
    out = Path(args.out) / "demo"
    df = render_demo(frames, X, found, probs, out, args.fps or fps)
    print(f"Processed {len(frames)} frames. Predicted-class counts:\n{df['pred_class'].value_counts().to_string()}")
    print(f"Saved in {out.resolve()}: demo.mp4, demo_predictions.csv, timeline.png")
    return 0


# --------------------------------------------------------------------------- #
def build_parser():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)

    def common(p, data_required=True):
        p.add_argument("--data", required=data_required, default=None,
                       help="folder with the UR Fall frames (or zips) and the urfall-cam0-*.csv files")
        p.add_argument("--out", default="fall_cnn_out", help="output folder (default: fall_cnn_out)")
        p.add_argument("--labels", nargs="*", default=None, help="explicit label CSV file(s)")
        p.add_argument("--mode", choices=["auto", "depth", "gray", "silhouette"], default="auto",
                       help="auto = depth if available else gray (default)")
        p.add_argument("--size", type=int, default=64, help="CNN input size (default 64)")
        p.add_argument("--thr", type=int, default=30, help="intensity threshold for --mode silhouette")
        p.add_argument("--depth-thr", type=float, default=120.0, dest="depth_thr",
                       help="person must be this many mm closer than the background (--mode depth)")
        p.add_argument("--min-area", type=int, default=150, dest="min_area")
        p.add_argument("--max-seqs", type=int, default=0, dest="max_seqs",
                       help="debug: use only N falls + N ADLs")
        p.add_argument("--seed", type=int, default=42)
        p.add_argument("--force", action="store_true", help="rebuild the cache")

    p = sub.add_parser("inspect", help="check dataset layout and labels")
    common(p)
    p.set_defaults(fn=cmd_inspect)

    p = sub.add_parser("preview", help="save a picture of preprocessed frames")
    common(p)
    p.add_argument("--seq", nargs="*", default=None, help="e.g. fall-03 adl-12")
    p.set_defaults(fn=cmd_preview)

    p = sub.add_parser("prepare", help="build the silhouette cache")
    common(p)
    p.set_defaults(fn=cmd_prepare)

    p = sub.add_parser("train", help="train, evaluate, export")
    common(p)
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch", type=int, default=64)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--patience", type=int, default=8)
    p.set_defaults(fn=cmd_train)

    p = sub.add_parser("demo", help="run the trained CNN on a sequence folder or a video")
    p.add_argument("--input", default=None, help="RGB frames folder or video file (used for the overlay)")
    p.add_argument("--depth-input", default=None, dest="depth_input",
                   help="folder with the cam0 depth PNGs (needed when the model was trained in depth mode)")
    p.add_argument("--depth-scale", type=float, default=6000.0, dest="depth_scale",
                   help="UR Fall depth constant C: 6000 for fall sequences, 7000 for ADL sequences")
    p.add_argument("--model", default="fall_cnn_out/model.pt")
    p.add_argument("--out", default="fall_cnn_out")
    p.add_argument("--fps", type=float, default=None)
    p.add_argument("--max-frames", type=int, default=3000, dest="max_frames")
    p.set_defaults(fn=cmd_demo)
    return ap


def main(argv=None) -> int:
    args = build_parser().parse_args(argv)
    return int(args.fn(args) or 0)


if __name__ == "__main__":
    sys.exit(main())

## 3. Accéder au dataset dans Google Drive

Créez un dossier `urfall` dans `Mon Drive` et placez-y les fichiers nécessaires.

Fichiers attendus :
- `urfall-cam0-falls.csv`
- `urfall-cam0-adls.csv`
- Les archives de profondeur caméra 0 : `fall-XX-cam0-d.zip` et `adl-XX-cam0-d.zip` (ou les dossiers d'images de profondeur déjà extraits).

**Les archives RGB ne sont pas nécessaires pour cet entraînement en mode `depth`.** Le notebook ne télécharge aucun fichier. Le programme extraira localement les archives ZIP que vous avez déjà placées dans ce dossier.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/urfall")
OUT_DIR = Path("/content/fall_cnn_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier dataset trouvé :", DATA_DIR.exists())
print("CSV trouvés :", [p.name for p in DATA_DIR.glob("*.csv")])
print("Archives de profondeur trouvées :", len(list(DATA_DIR.rglob("*-cam0-d.zip"))))

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Créez le dossier {DATA_DIR} dans Google Drive et ajoutez-y le dataset UR Fall."
    )

## 4. Vérifier le dataset

Cette étape vérifie les images de profondeur et les fichiers d'annotations. Elle extrait aussi les archives ZIP présentes dans votre dossier de données vers le dossier de sortie.

In [ ]:
!python fall_cnn.py inspect --data "{DATA_DIR}" --out "{OUT_DIR}" --mode depth

## 5. Entraîner le CNN

Le CNN classe chaque image en trois postures : `not_lying`, `falling` et `lying`.

À la fin, il :
- sauvegarde le modèle entraîné (`model.pt`) ;
- évalue le modèle sur des séquences non utilisées pendant l'entraînement ;
- exporte les prédictions image par image et les embeddings qui serviront d'entrées au RNN et au Random Forest.

La valeur par défaut est de 30 époques. Vous pouvez la modifier si nécessaire.

In [ ]:
!python fall_cnn.py train --data "{DATA_DIR}" --out "{OUT_DIR}" --mode depth --epochs 30

## 6. Consulter les résultats

In [ ]:
from IPython.display import Image, display
from pathlib import Path

report = OUT_DIR / "test_report.txt"
if report.exists():
    print(report.read_text())
else:
    print("Rapport de test non disponible.")

for filename in ["confusion_matrix.png", "training_curves.png"]:
    image_path = OUT_DIR / filename
    if image_path.exists():
        print(filename)
        display(Image(str(image_path)))

## 7. Télécharger les fichiers utiles

Le ZIP contient uniquement les fichiers à conserver et à transmettre pour la suite du projet. Il exclut le cache et les images extraites, qui peuvent être volumineux.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

files_to_save = [
    "model.pt",
    "test_report.txt",
    "metrics.json",
    "confusion_matrix.png",
    "training_curves.png",
    "frame_predictions.csv",
    "embeddings.npz",
]

zip_path = Path("/content/fall_cnn_results.zip")
with ZipFile(zip_path, "w", ZIP_DEFLATED) as z:
    for filename in files_to_save:
        file_path = OUT_DIR / filename
        if file_path.exists():
            z.write(file_path, arcname=filename)
        else:
            print("Fichier absent, ignoré :", filename)

print("Archive créée :", zip_path)
files.download(str(zip_path))